<a href="https://colab.research.google.com/github/Eng-Alaa-Mohamed-Ali/Eng-Alaa-Mohamed-Ali/blob/main/%D8%A8%D9%86%D8%A7%D8%A1_%D8%B4%D8%A8%D9%83%D8%A9_%D8%B9%D8%B5%D8%A8%D9%8A%D8%A9_%D9%84%D9%84%D8%AA%D8%B9%D8%B1%D9%81_%D8%B9%D9%84%D9%89_%D8%A3%D8%B1%D9%82%D8%A7%D9%85_%D9%85%D9%83%D8%AA%D9%88%D8%A8%D8%A9_%D8%A8%D8%AE%D8%B7_%D8%A7%D9%84%D9%8A%D8%AF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from PIL import Image

tf.compat.v1.disable_eager_execution()

In [4]:
#اختيار بيانات التدريب

mnist = tf.keras.datasets.mnist

In [6]:
#تنزيل بيانات التدريبوالاختبار
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [7]:
#طباعة عدد بيانات التدريبوالاختبار
print(len(x_train))
print(len(x_test))

60000
10000


In [8]:
x_train,x_test=x_train/255.0,x_test/255.0

In [11]:
import numpy as np

y_train = np.array(y_train).reshape(-1, 1)
y_test = np.array(y_test).reshape(-1, 1)
enc = OneHotEncoder(sparse_output=True)
enc.fit(y_train)
y_train = enc.transform(y_train)
y_test = enc.transform(y_test)

In [14]:
n_input = 784 # input layer (28x28 pixels)
n_hidden1 = 512 # 1st hidden layer
n_hidden2 = 256 # 2nd hidden layer
n_hidden3 = 128 # 3rd hidden layer
n_output = 10 # output layer (0-9 digits

In [15]:
learning_rate = 1e-4
n_iterations = 1000
batch_size = 128
dropout = 0.5

In [18]:
X = tf.compat.v1.placeholder("float", [None, n_input])
Y = tf.compat.v1.placeholder("float", [None, n_output])
keep_prob = tf.compat.v1.placeholder(tf.float32)

In [19]:
weights = {
    'w1': tf.Variable(tf.random.truncated_normal([n_input,
    n_hidden1], stddev=0.1)),
    'w2': tf.Variable(tf.random.truncated_normal([n_hidden1,
    n_hidden2], stddev=0.1)),
    'w3': tf.Variable(tf.random.truncated_normal([n_hidden2,
    n_hidden3], stddev=0.1)),
    'out': tf.Variable(tf.random.truncated_normal([n_hidden3,
    n_output], stddev=0.1)),
}

In [22]:
biases = {
    'b1': tf.Variable(tf.constant(0.1, shape=[n_hidden1])),
    'b2': tf.Variable(tf.constant(0.1, shape=[n_hidden2])),
    'b3': tf.Variable(tf.constant(0.1, shape=[n_hidden3])),
    'out': tf.Variable(tf.constant(0.1, shape=[n_output]))
}

In [23]:
layer_1 = tf.add(tf.matmul(X, weights['w1']), biases['b1'])
layer_2 = tf.add(tf.matmul(layer_1, weights['w2']), biases['b2'])
layer_3 = tf.add(tf.matmul(layer_2, weights['w3']), biases['b3'])
layer_drop = tf.nn.dropout(layer_3, keep_prob)
output_layer = tf.matmul(layer_3, weights['out']) + biases['out']

In [25]:
cross_entropy = tf.reduce_mean(
tf.nn.softmax_cross_entropy_with_logits(
labels=Y, logits=output_layer
))
train_step = tf.compat.v1.train.AdamOptimizer(1e-4).minimize(cross_entropy)

In [26]:
correct_pred = tf.equal(tf.argmax(output_layer, 1), tf.argmax(Y, 1))
accuracy = tf.reduce_mean(tf.cast(correct_pred, tf.float32))

In [27]:
init = tf.compat.v1.global_variables_initializer()
sess = tf.compat.v1.Session()
sess.run(init)

In [28]:
for i in range(n_iterations):
    startbatch = (i*batch_size) % len(x_train)
    endbatch = ((i+1)*batch_size) % len(x_train)
    batch_x = np.array(x_train[startbatch:endbatch])
    batch_x = batch_x.reshape(batch_size, -1)
    batch_y = y_train[startbatch:endbatch].toarray()
    if batch_x.shape != (128, 784):
        continue
    sess.run(train_step, feed_dict={
        X: batch_x, Y: (batch_y), keep_prob: dropout
    })

    if i % 100 == 0:
        minibatch_loss, minibatch_accuracy = sess.run(
            [cross_entropy, accuracy],
            feed_dict={X: batch_x, Y: batch_y, keep_prob: 1.0}
        )
        print(
            "Iteration",
            str(i),
            "\t| Loss =",
            str(minibatch_loss),
            "\t| Accuracy =",
            str(minibatch_accuracy)
        )

Iteration 0 	| Loss = 3.5948508 	| Accuracy = 0.203125
Iteration 100 	| Loss = 0.40800393 	| Accuracy = 0.8671875
Iteration 200 	| Loss = 0.32642964 	| Accuracy = 0.90625
Iteration 300 	| Loss = 0.34395123 	| Accuracy = 0.8984375
Iteration 400 	| Loss = 0.41038576 	| Accuracy = 0.875
Iteration 500 	| Loss = 0.34274074 	| Accuracy = 0.90625
Iteration 600 	| Loss = 0.24208081 	| Accuracy = 0.9140625
Iteration 700 	| Loss = 0.30180293 	| Accuracy = 0.9296875
Iteration 800 	| Loss = 0.6653559 	| Accuracy = 0.8671875
Iteration 900 	| Loss = 0.21191683 	| Accuracy = 0.9453125


In [29]:
x_test = x_test.reshape(-1,784)
test_accuracy = sess.run(accuracy, feed_dict={X: x_test, Y:
y_test.toarray(), keep_prob: 1.0})
print("\nAccuracy on test set:", test_accuracy)


Accuracy on test set: 0.9159


In [34]:
img = np.array(Image.open("/content/test_image.png").convert('L')).ravel()

In [36]:
prediction = sess.run(tf.argmax(output_layer, 1), feed_dict={X: [img]})
print ("Prediction for test image:", np.squeeze(prediction))

Prediction for test image: 4
